# 🖼️ Lesson 4.3 — Convolutional Neural Networks: How Machines See

**AI Crash Course for Absolute Beginners**

CNNs are specialised for grid data like images. In this notebook:
- See what convolution actually does to an image
- Build a CNN in PyTorch and train it on MNIST
- Visualise feature maps (what the network 'sees')
- Add batch normalisation and max-pooling

> Run each cell with **Shift + Enter**. GPU recommended but not required.

## ⚠️ GPU Recommended — Enable Before Running

Training a CNN on MNIST takes **10–15 minutes on CPU** but only **2–3 minutes on GPU**.

**Enable GPU now (free):**
1. Click **Runtime** in the top menu
2. Select **Change runtime type**
3. Set **Hardware accelerator** to **T4 GPU**
4. Click **Save**, then run from the top

> You only need to do this once per session. The cell below will confirm your device.

In [ ]:
!pip install torch torchvision matplotlib numpy --quiet

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader   # feeds data to the model in small batches
from torchvision import datasets, transforms
# torchvision.datasets — contains popular image datasets (MNIST, CIFAR-10, ImageNet, etc.)
# torchvision.transforms — image preprocessing tools (resize, crop, normalise, convert to tensor)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

---
## Part 1 — What Does Convolution Actually Do?

In [ ]:
from scipy.ndimage import convolve
from skimage import data as skdata

# Load a grayscale image (comes with scikit-image)
image = skdata.camera().astype(float) / 255.0   # 512x512 photographer

# Three classic kernels
kernels = {
    "Edge detection\n(Sobel-X)": np.array([[-1,0,1],[-2,0,2],[-1,0,1]]),
    "Blur\n(Gaussian 3x3)"     : np.ones((3,3)) / 9,
    "Sharpen"                  : np.array([[0,-1,0],[-1,5,-1],[0,-1,0]])
}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(image, cmap="gray")
axes[0].set_title("Original")
axes[0].axis("off")

for ax, (name, kernel) in zip(axes[1:], kernels.items()):
    filtered = convolve(image, kernel)
    filtered = np.clip(filtered, 0, 1)
    ax.imshow(filtered, cmap="gray")
    ax.set_title(name, fontsize=10)
    ax.axis("off")

plt.suptitle("Convolution with Different Kernels", fontsize=13)
plt.tight_layout()
plt.show()
print("In a CNN, the network learns these kernels automatically from data.")

---
## Part 2 — Load MNIST

In [ ]:
# transforms.Compose chains multiple preprocessing steps into one pipeline
# Every image goes through these steps in order before entering the model:
transform = transforms.Compose([
    transforms.ToTensor(),   # converts a PIL image (0-255 integers) to a PyTorch tensor (0.0-1.0 floats)
    # Normalize subtracts the mean and divides by the standard deviation for MNIST images
    # This centres the data around 0 — networks train faster on normalised input
    transforms.Normalize((0.1307,), (0.3081,))   # MNIST mean and std (precomputed from the full dataset)
])

# train=True loads training images, train=False loads test images
# download=True fetches the dataset from the internet if not already saved
train_data = datasets.MNIST(root="data", train=True,  download=True, transform=transform)
test_data  = datasets.MNIST(root="data", train=False, download=True, transform=transform)

# DataLoader wraps the dataset and serves it in batches
# batch_size=64 means the model sees 64 images at a time (one step of gradient descent per batch)
# shuffle=True randomises the order each epoch so the model doesn't memorise the sequence
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=256)   # larger batch for evaluation (no gradient needed)

print(f"Training images : {len(train_data):,}")
print(f"Test images     : {len(test_data):,}")
# Shape is (channels x height x width) — MNIST is 1 channel (grayscale), 28x28 pixels
print(f"Image shape     : {train_data[0][0].shape}  (channels x H x W)")

# Preview samples
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flatten()):
    img, label = train_data[i]
    ax.imshow(img.squeeze(), cmap="gray")   # squeeze() removes the channel dimension for display
    ax.set_title(label, fontsize=9)
    ax.axis("off")
plt.suptitle("MNIST Samples", fontsize=12)
plt.tight_layout()
plt.show()

---
## Part 3 — Build the CNN

In [ ]:
class MnistCNN(nn.Module):
    def __init__(self):
        super().__init__()

        # The network has two parts:
        # 1) Feature extractor: convolutional layers that detect edges, curves, shapes
        # 2) Classifier head: fully connected layers that map features to digit classes

        self.features = nn.Sequential(
            # Block 1: input is 1 channel x 28x28 pixels
            # nn.Conv2d(in_channels, out_channels, kernel_size)
            # 32 filters means the layer learns 32 different patterns to detect (edges, curves, etc.)
            # kernel_size=3 means each filter looks at a 3x3 patch of the image
            nn.Conv2d(1, 32, kernel_size=3),   # output: 32 x 26 x 26
            # BatchNorm2d normalises activations within each batch — stabilises and speeds up training
            nn.BatchNorm2d(32),
            nn.ReLU(),
            # MaxPool2d(2) takes the maximum value in each 2x2 patch, halving the spatial size
            # This makes the model less sensitive to exact position of features
            nn.MaxPool2d(2),                   # output: 32 x 13 x 13

            # Block 2: deeper features, more filters
            nn.Conv2d(32, 64, kernel_size=3),  # output: 64 x 11 x 11
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),                   # output: 64 x 5 x 5
        )

        self.classifier = nn.Sequential(
            # Flatten turns the 3D tensor (64 x 5 x 5) into a 1D vector (1600 values)
            # so we can feed it into a standard fully connected (Linear) layer
            nn.Flatten(),
            nn.Linear(64 * 5 * 5, 128),   # 64*5*5=1600 inputs → 128 neurons
            nn.ReLU(),
            nn.Dropout(0.3),   # randomly turns off 30% of neurons during training to prevent overfitting
            nn.Linear(128, 10)  # 10 output neurons — one for each digit class (0-9)
        )

    def forward(self, x):
        x = self.features(x)        # extract visual features
        return self.classifier(x)   # classify into one of 10 digits


model = MnistCNN().to(device)
print(model)
print(f"\nParameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

---
## Part 4 — Train the CNN

In [ ]:
criterion = nn.CrossEntropyLoss()   # combines softmax + log loss — standard for multi-class classification
optimizer = optim.Adam(model.parameters(), lr=1e-3)
# StepLR reduces the learning rate by multiplying by gamma every step_size epochs
# After epoch 3: lr = 1e-3 * 0.5 = 0.0005 | After epoch 6: lr = 0.00025
# This helps fine-tune later in training without overshooting the minimum
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

EPOCHS = 8
train_accs, test_accs = [], []

for epoch in range(EPOCHS):
    # === TRAINING ===
    model.train()   # enables Dropout and BatchNorm training mode
    correct = total = 0
    for X_batch, y_batch in train_loader:
        # .to(device) moves data to GPU if available (must match the model's device)
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
        # .argmax(dim=1) picks the class with the highest score (the model's prediction)
        # dim=1 means "take the max across the class dimension" (each row = one image)
        preds = model(X_batch).argmax(dim=1)
        correct += (preds == y_batch).sum().item()   # .item() converts tensor to a Python int
        total   += len(y_batch)
    train_accs.append(correct / total)

    # === EVALUATION ===
    model.eval()   # disables Dropout so all neurons are active during testing
    correct = total = 0
    with torch.no_grad():   # skip gradient calculations — saves memory and speeds up evaluation
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            preds = model(X_batch).argmax(dim=1)
            correct += (preds == y_batch).sum().item()
            total   += len(y_batch)
    test_accs.append(correct / total)
    scheduler.step()   # reduce learning rate if it's a step epoch

    print(f"Epoch {epoch+1}/{EPOCHS}  train={train_accs[-1]:.3f}  test={test_accs[-1]:.3f}")

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(1, EPOCHS+1), train_accs, label="Train", marker="o", color="#1a6bc8")
plt.plot(range(1, EPOCHS+1), test_accs,  label="Test",  marker="s", color="#c8401a")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title(f"CNN on MNIST — Final test accuracy: {test_accs[-1]:.1%}")
plt.legend()
plt.tight_layout()
plt.show()

---
## Part 5 — Visualise Feature Maps

In [ ]:
# Pick one test image
sample_img, sample_label = test_data[7]
# .unsqueeze(0) adds a batch dimension: shape (1, 28, 28) → (1, 1, 28, 28)
# Models expect input as (batch_size, channels, H, W) — even for a single image
sample_img_batch = sample_img.unsqueeze(0).to(device)

# A "hook" is a function that PyTorch calls automatically during the forward pass
# We use it to capture and save the intermediate activations (feature maps) from a layer
activations = {}
def hook_fn(name):
    # This inner function is called every time the registered layer processes input
    # .detach() removes gradient tracking (we only want to view, not train)
    # .cpu() moves the tensor back from GPU to CPU for display
    def fn(module, input, output):
        activations[name] = output.detach().cpu()
    return fn

# register_forward_hook attaches our hook to the first conv layer
# It will fire every time data passes through that layer
model.features[0].register_forward_hook(hook_fn("conv1"))
model.features[4].register_forward_hook(hook_fn("conv2"))

model.eval()
with torch.no_grad():
    out = model(sample_img_batch)
    pred = out.argmax(dim=1).item()   # .item() extracts the integer value from a 1-element tensor

print(f"True label: {sample_label}  |  Predicted: {pred}")

# Plot first 16 feature maps from conv1
# Each feature map shows what pattern that particular filter activated on
feature_maps = activations["conv1"].squeeze()   # remove batch dimension
fig, axes = plt.subplots(2, 8, figsize=(16, 5))
axes[0][0].imshow(sample_img.squeeze(), cmap="gray")
axes[0][0].set_title(f"Input (digit {sample_label})")
axes[0][0].axis("off")

for i, ax in enumerate(axes.flatten()[1:15]):
    ax.imshow(feature_maps[i], cmap="viridis")   # viridis = colour scale from purple to yellow
    ax.set_title(f"Filter {i+1}")
    ax.axis("off")

plt.suptitle("Conv Layer 1 Feature Maps", fontsize=13)
plt.tight_layout()
plt.show()

---
## Challenge Exercises

1. **CIFAR-10**: Replace MNIST with `datasets.CIFAR10`. The images are 32x32x3. Adjust the first Conv layer to accept 3 channels (`nn.Conv2d(3, 32, 3)`).
2. **More filters**: Double the filter count to 64 in the first block. How many more parameters does this add?
3. **Wrong predictions**: Find test images the model got wrong. Do they look ambiguous to you too?

---
**Next lesson:** [Lesson 4.4 — Transformers and Attention](https://colab.research.google.com/github/GirlEf/ai-crash-course/blob/main/notebooks/lesson-4.4-transformers.ipynb)